In [ ]:
# Copyright 2026 Google LLC
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#     https://www.apache.org/licenses/LICENSE-2.0
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 02 · The two pieces, in isolation

Before bridging anything, prove each half works on its own:

- **Part 1** — the raw MCP Python SDK connects to the managed BigQuery MCP
  endpoint (remote, streamable HTTP, bearer auth), lists tools, and calls one
  read-only tool.
- **Part 2** — `AnthropicVertex` sends a trivial message to Claude on Vertex.

No agent and no bridge yet. Reusable connect/list/call logic lives in
`src/mcp_client.py` so this notebook stays thin.

In [ ]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()
sys.path.append(os.path.abspath("src"))

from mcp_client import bq_mcp_session

MODEL_ID = os.environ["CLAUDE_MODEL_ID"]
REGION = os.environ["GOOGLE_CLOUD_REGION"]
PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]

## Part 1 — the MCP half

Open a session, list the tools the server exposes, then call one read-only
tool. We discover tools first rather than hardcoding a name: we pick a tool
that requires no arguments so the call is safe and reproducible regardless of
the server's current tool surface.

In [ ]:
async with bq_mcp_session() as session:
    listing = await session.list_tools()
    print("Tools exposed by the BigQuery MCP server:")
    for t in listing.tools:
        required = (t.inputSchema or {}).get("required", [])
        print(f"  - {t.name}  (required args: {required})")

    # Call one read-only tool directly. list_dataset_ids needs only a projectId;
    # we point it at the public data project, so no private data is touched.
    print("\nCalling read-only tool: list_dataset_ids on bigquery-public-data")
    result = await session.call_tool("list_dataset_ids", {"projectId": "bigquery-public-data"})
    text = "\n".join(b.text for b in result.content if getattr(b, "type", None) == "text")
    print("Result (truncated):\n", text[:1000])

## Part 2 — the Vertex half

A trivial round-trip to Claude on Vertex via `AnthropicVertex` (ADC auth).

In [ ]:
from anthropic import AnthropicVertex

client = AnthropicVertex(project_id=PROJECT, region=REGION)
msg = client.messages.create(
    model=MODEL_ID,
    max_tokens=64,
    messages=[{"role": "user", "content": "In one sentence, what is BigQuery?"}],
)
print("".join(b.text for b in msg.content if b.type == "text"))

## Recap

- ✅ MCP endpoint reachable, tools listed, one read-only tool called.
- ✅ Claude on Vertex responded.

Two green checkmarks. Notebook 03 wires these halves together by hand.

## Cleanup

Nothing to clean up: the MCP session is opened with `async with` and closes
automatically, and this notebook creates no persistent cloud resources.